# 🧠 Robotic Arm Quality Control - Fixed CNN Training
### Fixes Applied:
- ✅ Dataset leakage prevention (separate fresh/rotten sources)
- ✅ Apples removed (grapes, strawberries, berries only)
- ✅ Class balancing (good ≈ defective)
- ✅ Dropout(0.5) added
- ✅ Proper train/val split (no leakage)
- ✅ EarlyStopping + ReduceLROnPlateau
- ✅ Kaggle token via secrets (not hardcoded)
- ✅ Class weights for imbalance
- ✅ Confusion matrix + classification report
- ✅ TFLite export for edge deployment

In [ ]:
# ============================================================
# CELL 1: Setup & Dependencies
# ============================================================
import os, shutil, random, zipfile
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

!pip install -q kagglehub scikit-learn seaborn
import kagglehub

# ⚠️  SECURITY FIX: Use Colab Secrets instead of hardcoding token
# In Colab: Secrets tab (🔑) → Add KAGGLE_API_TOKEN secret
try:
    from google.colab import userdata
    os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
    print('✅ Kaggle token loaded from Colab Secrets')
except Exception:
    print('⚠️  Colab Secrets not available - set KAGGLE_API_TOKEN manually')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'✅ TF version: {tf.__version__}')
print(f'✅ GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ============================================================
# CELL 2: Download Fruits360 (Good fruits ONLY - no apples)
# ============================================================
print('📥 Downloading Fruits 360...')
path_fruits360 = kagglehub.dataset_download('moltean/fruits')
print(f'Fruits 360 path: {path_fruits360}')

# FIX: Only project-relevant fruits (NO APPLES)
GOOD_FRUITS = [
    'Strawberry',
    'Grape Blue',
    'Grape Pink',
    'Grape White',
    'Raspberry',
    'Blueberry'
]

print(f'✅ Target fruits (good class): {GOOD_FRUITS}')

In [ ]:
# ============================================================
# CELL 3: Unzip Mendeley/Rotten Dataset
# ============================================================
# 🛑 STOP: Upload your rotten fruits ZIP to Colab sidebar first!
zip_files = [f for f in os.listdir('.') if f.endswith('.zip')]

if zip_files:
    zip_path = zip_files[0]
    print(f'📦 Found: {zip_path}. Unzipping...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('mendeley_dataset')
    print('✅ Dataset unzipped!')
else:
    raise FileNotFoundError('❌ No ZIP found! Upload your rotten/defective fruits ZIP to Colab.')

In [ ]:
# ============================================================
# CELL 4: Build Balanced, Leak-Free Dataset
# ============================================================
BASE_DIR     = './dataset'
TRAIN_DIR    = os.path.join(BASE_DIR, 'train')
VAL_DIR      = os.path.join(BASE_DIR, 'val')

for split in ['train', 'val']:
    for cls in ['good', 'defective']:
        os.makedirs(os.path.join(BASE_DIR, split, cls), exist_ok=True)

# --- Collect good images from Fruits360 (no apples) ---
good_images = []
f360_train = None
for root, dirs, files in os.walk(path_fruits360):
    if 'Training' in dirs:
        f360_train = os.path.join(root, 'Training')
        break

if f360_train:
    for fruit in GOOD_FRUITS:
        fruit_path = os.path.join(f360_train, fruit)
        if os.path.exists(fruit_path):
            imgs = [os.path.join(fruit_path, f) for f in os.listdir(fruit_path) if f.lower().endswith('.jpg')]
            good_images.extend(imgs)
            print(f'  {fruit}: {len(imgs)} images')

# --- Collect defective images from Mendeley ---
defective_images = []
mendeley_path = './mendeley_dataset'
for root, dirs, files in os.walk(mendeley_path):
    folder = os.path.basename(root).lower()
    if 'rotten' in folder or 'defect' in folder or 'bad' in folder:
        imgs = [os.path.join(root, f) for f in files if f.lower().endswith(('.jpg', '.png'))]
        defective_images.extend(imgs)

print(f'\nRaw good images: {len(good_images)}')
print(f'Raw defective images: {len(defective_images)}')

# --- FIX: Balance classes ---
MAX_PER_CLASS = min(len(good_images), len(defective_images), 600)
random.shuffle(good_images)
random.shuffle(defective_images)
good_images     = good_images[:MAX_PER_CLASS]
defective_images = defective_images[:MAX_PER_CLASS]

print(f'\n✅ Balanced — Good: {len(good_images)} | Defective: {len(defective_images)}')

# --- FIX: Proper train/val split BEFORE copying (prevents leakage) ---
def split_and_copy(image_list, class_name, train_ratio=0.8):
    random.shuffle(image_list)
    split = int(len(image_list) * train_ratio)
    train_imgs = image_list[:split]
    val_imgs   = image_list[split:]
    for src in train_imgs:
        dst = os.path.join(TRAIN_DIR, class_name, f'{class_name}_{os.path.basename(src)}')
        shutil.copy(src, dst)
    for src in val_imgs:
        dst = os.path.join(VAL_DIR, class_name, f'{class_name}_{os.path.basename(src)}')
        shutil.copy(src, dst)
    return len(train_imgs), len(val_imgs)

tr_g, va_g = split_and_copy(good_images, 'good')
tr_d, va_d = split_and_copy(defective_images, 'defective')

print(f'\nTrain → good: {tr_g} | defective: {tr_d}')
print(f'Val   → good: {va_g} | defective: {va_d}')
print('✅ No augmentation leakage: train/val split done BEFORE generators!')

In [ ]:
# ============================================================
# CELL 5: Data Generators (separate dirs = no leakage)
# ============================================================
IMG_SIZE   = (128, 128)   # Faster training; good enough for MVP
BATCH_SIZE = 32

# Training: augmentation ON
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],   # Handles lighting variation
    shear_range=0.1
)

# Validation: NO augmentation (only rescale)
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True,
    seed=SEED
)

val_gen = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

print('Class indices:', train_gen.class_indices)
print(f'Train batches: {len(train_gen)} | Val batches: {len(val_gen)}')

In [ ]:
# ============================================================
# CELL 6: Visualize Sample Images (sanity check)
# ============================================================
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
batch_x, batch_y = next(train_gen)
for i, ax in enumerate(axes.flatten()):
    if i < len(batch_x):
        ax.imshow(batch_x[i])
        label = 'good' if batch_y[i] == train_gen.class_indices.get('good', 0) else 'defective'
        ax.set_title(label, color='green' if label == 'good' else 'red')
        ax.axis('off')
plt.suptitle('🔍 Sample Training Images (verify no background bias!)', fontsize=13)
plt.tight_layout()
plt.show()
print('\n👀 CHECK: Do defective images look actually rotten/bruised?')
print('👀 CHECK: Do good images have varied backgrounds (not all white)?')

In [ ]:
# ============================================================
# CELL 7: Build CNN with Dropout (fixes overfitting)
# ============================================================
print('🏗️  Building CNN with Dropout...')

model = models.Sequential([
    # Block 1
    layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(128, 128, 3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),

    # Block 2
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),

    # Block 3
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),

    # Classifier head
    layers.GlobalAveragePooling2D(),   # Better than Flatten for generalization
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),               # FIX: prevents overfitting
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# ============================================================
# CELL 8: Train with EarlyStopping & Class Weights
# ============================================================
from sklearn.utils.class_weight import compute_class_weight

# Compute class weights to handle any residual imbalance
labels = train_gen.classes
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)
class_weight_dict = dict(enumerate(class_weights_arr))
print(f'Class weights: {class_weight_dict}')

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        'best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print('🔥 Starting Training...')
history = model.fit(
    train_gen,
    epochs=30,
    validation_data=val_gen,
    class_weight=class_weight_dict,
    callbacks=callbacks
)

print(f'\n✅ Training stopped at epoch {len(history.history["accuracy"])}')

In [ ]:
# ============================================================
# CELL 9: Training Curves
# ============================================================
acc     = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss    = history.history['loss']
val_loss = history.history['val_loss']
epochs_ran = range(1, len(acc)+1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_ran, acc,     'b-o', label='Train Acc')
ax1.plot(epochs_ran, val_acc, 'r-o', label='Val Acc')
ax1.set_title('Accuracy')
ax1.legend()
ax1.set_xlabel('Epoch')

ax2.plot(epochs_ran, loss,     'b-o', label='Train Loss')
ax2.plot(epochs_ran, val_loss, 'r-o', label='Val Loss')
ax2.set_title('Loss')
ax2.legend()
ax2.set_xlabel('Epoch')

plt.suptitle('Training History', fontsize=14)
plt.tight_layout()
plt.show()

# Bias detection
gap = max(acc) - max(val_acc)
print(f'\n🔍 Overfitting gap (train-val acc): {gap:.3f}')
if gap > 0.15:
    print('⚠️  WARNING: High overfitting detected. Consider more data or stronger regularization.')
else:
    print('✅ Overfitting looks manageable.')

In [ ]:
# ============================================================
# CELL 10: Confusion Matrix + Classification Report
# ============================================================
print('📊 Evaluating on validation set...')
val_gen.reset()
y_pred_probs = model.predict(val_gen, verbose=1)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()
y_true = val_gen.classes

class_names = list(val_gen.class_indices.keys())

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# Classification Report
print('\n' + classification_report(y_true, y_pred, target_names=class_names))

# Bias check: if one class dominates predictions
unique, counts = np.unique(y_pred, return_counts=True)
pred_dist = dict(zip(unique, counts))
print(f'Prediction distribution: {pred_dist}')
total = len(y_pred)
for cls_id, cnt in pred_dist.items():
    pct = cnt / total * 100
    if pct > 75:
        print(f'⚠️  BIAS DETECTED: Model predicts class {cls_id} ({class_names[cls_id]}) {pct:.1f}% of the time!')
        print('   → Likely cause: class imbalance or dataset leakage.')
    else:
        print(f'✅ Class {class_names[cls_id]}: {pct:.1f}% predictions — OK')

In [ ]:
# ============================================================
# CELL 11: Cross-Validation (K-Fold estimate on val set)
# ============================================================
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

print('🔁 Extracting features for cross-validation check...')

# Use the CNN as feature extractor (remove final sigmoid layer)
feature_model = models.Model(
    inputs=model.input,
    outputs=model.layers[-3].output  # output before dropout
)

val_gen.reset()
features = feature_model.predict(val_gen, verbose=1)
labels_cv = val_gen.classes

# Stratified 5-fold cross-validation on extracted features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_scaled, labels_cv)):
    clf = LogisticRegression(max_iter=1000, random_state=SEED)
    clf.fit(X_scaled[tr_idx], labels_cv[tr_idx])
    score = clf.score(X_scaled[va_idx], labels_cv[va_idx])
    cv_scores.append(score)
    print(f'  Fold {fold+1}: {score:.4f}')

print(f'\n✅ Cross-val accuracy: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}')
if np.std(cv_scores) > 0.08:
    print('⚠️  High variance across folds — model may be unstable. Need more data.')
else:
    print('✅ Low variance — model is consistent.')

In [ ]:
# ============================================================
# CELL 12: Export — .h5 + TFLite
# ============================================================
from google.colab import files

# Load the best checkpoint
model.load_weights('best_model.h5')

# Save full model
model.save('robotic_arm_classifier_v2.h5')
print('✅ Full model saved: robotic_arm_classifier_v2.h5')

# TFLite export for ESP32/Raspberry Pi
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]   # INT8 quantization
tflite_model = converter.convert()

with open('robotic_arm_classifier_v2.tflite', 'wb') as f:
    f.write(tflite_model)

tflite_size = os.path.getsize('robotic_arm_classifier_v2.tflite') / 1024
print(f'✅ TFLite model saved ({tflite_size:.1f} KB) — ready for edge deployment!')

files.download('robotic_arm_classifier_v2.h5')
files.download('robotic_arm_classifier_v2.tflite')

## ✅ Post-Training Checklist

| Check | Target | Action |
|-------|--------|--------|
| Val Accuracy | > 85% | If not, add more real-world good images |
| Overfitting gap | < 15% | If high, increase Dropout or add more data |
| Confusion matrix | Both classes balanced | If biased, check dataset balance |
| CV std deviation | < 0.08 | If high, dataset too small or noisy |
| Prediction distribution | 40-60% each class | If skewed, class imbalance remains |

## ⚠️ Known Remaining Risks
1. **Fruits360 background leakage**: Studio white BG still present — mitigated by `brightness_range` augmentation but not eliminated. Future: replace with real-world fresh fruit images.
2. **No true test set**: Validation doubles as test. Add 20% holdout set when dataset > 2000 images.
3. **Duplicate images**: Not filtered. Run `imagededup` library if accuracy seems too high.